# Set Up

In [1]:
!pip install -U sentence-transformers

In [2]:
# This cell will authenticate you and mount your Drive in the Colab.
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
from sentence_transformers import SentenceTransformer
import pandas as pd
import torch

In [4]:
# Load a pretrained Sentence Transformer model
model = SentenceTransformer("all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [5]:
# Read in the datasets made in other code files
perenual_data = pd.read_csv('./drive/MyDrive/DATASCI 266/Project/transformed_wiki_descriptions_with_perenual_data_text.csv')

In [6]:
generated_data = pd.read_csv('./drive/MyDrive/DATASCI 266/Project/transformed_wiki_descriptions_with_generated_descriptions.csv')

In [7]:
class mapping_class:
  def __init__(self, d):
    self._dict = d
  def mapping_func(self, ind):
    return [self._dict[i].lower() for i in ind]

my_map = mapping_class(perenual_data['name'].to_dict())
# my_map.mapping_func(torch.topk(similarities_two, k=10, dim=1).indices.tolist()[0])

# Running Base Model

In [8]:
perenual_data.columns

Index(['user_description', 'name', 'wiki_description', 'perenual_data_text'], dtype='object')

In [9]:
# Calculate embeddings by calling model.encode()
embeddings = model.encode(perenual_data['perenual_data_text'])
print(embeddings.shape)
# [3000, 384]

(1174, 384)


In [10]:
wiki_embeddings = model.encode(perenual_data['user_description'])
print(wiki_embeddings.shape)
# [3000, 384]

(1174, 384)


In [11]:
# Calculate the embedding similarities
similarities = model.similarity(wiki_embeddings, embeddings)
print(similarities)

tensor([[0.5455, 0.4124, 0.5940,  ..., 0.3559, 0.3507, 0.3918],
        [0.4395, 0.5067, 0.3889,  ..., 0.2615, 0.2922, 0.2462],
        [0.4697, 0.4765, 0.5114,  ..., 0.2575, 0.3185, 0.3357],
        ...,
        [0.1890, 0.2943, 0.2405,  ..., 0.3204, 0.4652, 0.3524],
        [0.1934, 0.2575, 0.2877,  ..., 0.4406, 0.5284, 0.4242],
        [0.2312, 0.2951, 0.2740,  ..., 0.4403, 0.4318, 0.4739]])


In [12]:
print(similarities.shape)

torch.Size([1174, 1174])


In [14]:
# Use the similarity to find the predicted plant value
results = pd.DataFrame(perenual_data['name'][torch.argmax(similarities, dim=1).tolist()])
results.rename(columns={'name': 'predicted'}, inplace=True)
results['predicted'] = results['predicted'].str.lower()
results.reset_index(inplace=True, drop=True)
results['actual'] = perenual_data['name'].str.lower()
results['predicted_top_5'] = [i for i in map(my_map.mapping_func, torch.topk(similarities, k=5, dim=1).indices.tolist())]
results['predicted_top_10'] = [i for i in map(my_map.mapping_func, torch.topk(similarities, k=10, dim=1).indices.tolist())]
results

,predicted,actual,predicted_top_5,predicted_top_10
0,fraser fir,european silver fir,"[fraser fir, caucasian fir, alpine fir, mounta...","[fraser fir, caucasian fir, alpine fir, mounta..."
1,grand fir,big leaf maple,"[grand fir, fraser fir, hornbeam maple, tree h...","[grand fir, fraser fir, hornbeam maple, tree h..."
2,fraser fir,alpine fir,"[fraser fir, tree heath, norway spruce, grand ...","[fraser fir, tree heath, norway spruce, grand ..."
3,fraser fir,noble fir,"[fraser fir, mountain mahogany, mountain dogwo...","[fraser fir, mountain mahogany, mountain dogwo..."
4,tree heath,amur maple,"[tree heath, grand fir, silver birch, rain tre...","[tree heath, grand fir, silver birch, rain tre..."
...,...,...,...,...
1169,weeping beech,fokienia,"[weeping beech, norway spruce, grand fir, amer...","[weeping beech, norway spruce, grand fir, amer..."
1170,lanceleaf coreopsis,forsythia,"[lanceleaf coreopsis, spring beauty, candle la...","[lanceleaf coreopsis, spring beauty, candle la..."
1171,joe pye weed,korean forsythia,"[joe pye weed, winter begonia, cardinal larksp...","[joe pye weed, winter begonia, cardinal larksp..."
1172,cardinal larkspur,greenstem forsythia,"[cardinal larkspur, tall bellflower, spring be...","[cardinal larkspur, tall bellflower, spring be..."


In [15]:
# Calculating accuracy
print(f'top 1: {sum(results['predicted'] == results['actual']) / results.shape[0]}')
print(f'top 5: {sum([results['actual'][i] in results['predicted_top_5'][i] for i in range(results.shape[0])]) / results.shape[0]}')
print(f'top 10: {sum([results['actual'][i] in results['predicted_top_10'][i] for i in range(results.shape[0])]) / results.shape[0]}')

top 1: 0.05110732538330494
top 5: 0.13287904599659284
top 10: 0.18824531516183987


# Running Finetuned Model

In [16]:
from datasets import load_dataset
from datasets import Dataset
from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
    SentenceTransformerModelCardData,
)
from sentence_transformers.losses import CoSENTLoss
from sentence_transformers.training_args import BatchSamplers
from sentence_transformers.evaluation import TripletEvaluator
import os

In [17]:
# Restructure the data for training
joint_df = pd.merge(perenual_data, generated_data, how='outer', right_on = 'name',left_on = 'name')

In [18]:
temp = joint_df['generated_description'].to_list()

In [19]:
temp2 = temp[5:] + temp[:5] # rotate the descriptions around 5 to get some false matches

In [20]:
# temp[0]

In [21]:
# temp2[0]

In [22]:
train_dataset = Dataset.from_dict({
    'sentence1': joint_df['perenual_data_text'].to_list() + joint_df['perenual_data_text'].to_list(),
    'sentence2': joint_df['generated_description'].to_list() + temp2,
    # Scores are typically in a 0 to 1 or 0 to 5 range, normalized to 0-1 for this loss function
    'score': [1] * joint_df.shape[0] + [0] * joint_df.shape[0]
})

In [23]:
# Start with a good pre-trained S-BERT model
model_name = 'all-MiniLM-L6-v2'
model_two = SentenceTransformer(model_name)

In [24]:
os.environ["WANDB_DISABLED"] = "true"

In [25]:
# # look at the layers in the model
# for i in model_two.named_parameters():
#   print(i[0])

In [26]:
# Define a loss function
loss = CoSENTLoss(model_two)

# Specify training args
args = SentenceTransformerTrainingArguments(
    # Required parameter:
    output_dir="models/fine-tuned-sbert",
    # Optional training parameters:
    num_train_epochs=1,
    per_device_train_batch_size=16,
    warmup_ratio=0.1,
    fp16=True,  # Set to False if GPU can't handle FP16
    bf16=False,  # Set to True if GPU supports BF16
    batch_sampler=BatchSamplers.NO_DUPLICATES
)

#freeze all layers except the last layer
for name, param in model_two.named_parameters():
    if not 'layer.5' in name:
        param.requires_grad = False

# Train
trainer = SentenceTransformerTrainer(
    model=model_two,
    args=args,
    train_dataset=train_dataset,
    loss=loss
)
trainer.train()

# Save the trained model
model_two.save_pretrained("models/fine-tuned-sbert/final")


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss


In [27]:
# Calculate embeddings by calling model.encode()
embeddings_two = model_two.encode(perenual_data['perenual_data_text'])
print(embeddings_two.shape)
# [3000, 384]

(1174, 384)


In [28]:
wiki_embeddings_two = model_two.encode(perenual_data['user_description'])

In [29]:
# Calculate the embedding similarities
similarities_two = model_two.similarity(wiki_embeddings_two, embeddings_two)
print(similarities)

tensor([[0.5455, 0.4124, 0.5940,  ..., 0.3559, 0.3507, 0.3918],
        [0.4395, 0.5067, 0.3889,  ..., 0.2615, 0.2922, 0.2462],
        [0.4697, 0.4765, 0.5114,  ..., 0.2575, 0.3185, 0.3357],
        ...,
        [0.1890, 0.2943, 0.2405,  ..., 0.3204, 0.4652, 0.3524],
        [0.1934, 0.2575, 0.2877,  ..., 0.4406, 0.5284, 0.4242],
        [0.2312, 0.2951, 0.2740,  ..., 0.4403, 0.4318, 0.4739]])


In [30]:
# Use the similarity to find the predicted plant value
results_two = pd.DataFrame(perenual_data['name'][torch.argmax(similarities_two, dim=1).tolist()])
results_two.rename(columns={'name': 'predicted'}, inplace=True)
results_two['predicted'] = results_two['predicted'].str.lower()
results_two.reset_index(inplace=True, drop=True)
results_two['actual'] = perenual_data['name'].str.lower()
results_two['predicted_top_5'] = [i for i in map(my_map.mapping_func, torch.topk(similarities_two, k=5, dim=1).indices.tolist())]
results_two['predicted_top_10'] = [i for i in map(my_map.mapping_func, torch.topk(similarities_two, k=10, dim=1).indices.tolist())]
results_two

,predicted,actual,predicted_top_5,predicted_top_10
0,fraser fir,european silver fir,"[fraser fir, caucasian fir, alpine fir, mounta...","[fraser fir, caucasian fir, alpine fir, mounta..."
1,grand fir,big leaf maple,"[grand fir, tree heath, hornbeam maple, fraser...","[grand fir, tree heath, hornbeam maple, fraser..."
2,fraser fir,alpine fir,"[fraser fir, tree heath, grand fir, norway spr...","[fraser fir, tree heath, grand fir, norway spr..."
3,mountain palm,noble fir,"[mountain palm, mountain mahogany, alps yarrow...","[mountain palm, mountain mahogany, alps yarrow..."
4,tree heath,amur maple,"[tree heath, grand fir, chehalis apple, hornbe...","[tree heath, grand fir, chehalis apple, hornbe..."
...,...,...,...,...
1169,norway spruce,fokienia,"[norway spruce, weeping beech, grand fir, amur...","[norway spruce, weeping beech, grand fir, amur..."
1170,lanceleaf coreopsis,forsythia,"[lanceleaf coreopsis, spring beauty, dahlia, y...","[lanceleaf coreopsis, spring beauty, dahlia, y..."
1171,joe pye weed,korean forsythia,"[joe pye weed, winter begonia, blue ginger, ph...","[joe pye weed, winter begonia, blue ginger, ph..."
1172,cardinal larkspur,greenstem forsythia,"[cardinal larkspur, tall bellflower, decorativ...","[cardinal larkspur, tall bellflower, decorativ..."


In [31]:
# Calculating accuracy
print(f'top 1: {sum(results_two['predicted'] == results_two['actual']) / results_two.shape[0]}')
print(f'top 5: {sum([results_two['actual'][i] in results_two['predicted_top_5'][i] for i in range(results_two.shape[0])]) / results_two.shape[0]}')
print(f'top 10: {sum([results_two['actual'][i] in results_two['predicted_top_10'][i] for i in range(results_two.shape[0])]) / results_two.shape[0]}')


top 1: 0.05025553662691652
top 5: 0.141396933560477
top 10: 0.2018739352640545


# Looking into missed values

In [32]:
# Looking into the ones that were off
results_two[results_two['predicted'] != results_two['actual']]

,predicted,actual,predicted_top_5,predicted_top_10
0,fraser fir,european silver fir,"[fraser fir, caucasian fir, alpine fir, mounta...","[fraser fir, caucasian fir, alpine fir, mounta..."
1,grand fir,big leaf maple,"[grand fir, tree heath, hornbeam maple, fraser...","[grand fir, tree heath, hornbeam maple, fraser..."
2,fraser fir,alpine fir,"[fraser fir, tree heath, grand fir, norway spr...","[fraser fir, tree heath, grand fir, norway spr..."
3,mountain palm,noble fir,"[mountain palm, mountain mahogany, alps yarrow...","[mountain palm, mountain mahogany, alps yarrow..."
4,tree heath,amur maple,"[tree heath, grand fir, chehalis apple, hornbe...","[tree heath, grand fir, chehalis apple, hornbe..."
...,...,...,...,...
1169,norway spruce,fokienia,"[norway spruce, weeping beech, grand fir, amur...","[norway spruce, weeping beech, grand fir, amur..."
1170,lanceleaf coreopsis,forsythia,"[lanceleaf coreopsis, spring beauty, dahlia, y...","[lanceleaf coreopsis, spring beauty, dahlia, y..."
1171,joe pye weed,korean forsythia,"[joe pye weed, winter begonia, blue ginger, ph...","[joe pye weed, winter begonia, blue ginger, ph..."
1172,cardinal larkspur,greenstem forsythia,"[cardinal larkspur, tall bellflower, decorativ...","[cardinal larkspur, tall bellflower, decorativ..."


In [33]:
perenual_data[perenual_data['name'].str.lower()=='palmatifidum japanese maple*']

,user_description,name,wiki_description,perenual_data_text


In [34]:
results_two[[not (results_two['actual'][i] in results_two['predicted_top_10'][i]) for i in range(results_two.shape[0])]]

,predicted,actual,predicted_top_5,predicted_top_10
1,grand fir,big leaf maple,"[grand fir, tree heath, hornbeam maple, fraser...","[grand fir, tree heath, hornbeam maple, fraser..."
2,fraser fir,alpine fir,"[fraser fir, tree heath, grand fir, norway spr...","[fraser fir, tree heath, grand fir, norway spr..."
3,mountain palm,noble fir,"[mountain palm, mountain mahogany, alps yarrow...","[mountain palm, mountain mahogany, alps yarrow..."
4,tree heath,amur maple,"[tree heath, grand fir, chehalis apple, hornbe...","[tree heath, grand fir, chehalis apple, hornbe..."
5,sugar maple,fullmoon maple,"[sugar maple, japanese maple, big leaf maple, ...","[sugar maple, japanese maple, big leaf maple, ..."
...,...,...,...,...
1169,norway spruce,fokienia,"[norway spruce, weeping beech, grand fir, amur...","[norway spruce, weeping beech, grand fir, amur..."
1170,lanceleaf coreopsis,forsythia,"[lanceleaf coreopsis, spring beauty, dahlia, y...","[lanceleaf coreopsis, spring beauty, dahlia, y..."
1171,joe pye weed,korean forsythia,"[joe pye weed, winter begonia, blue ginger, ph...","[joe pye weed, winter begonia, blue ginger, ph..."
1172,cardinal larkspur,greenstem forsythia,"[cardinal larkspur, tall bellflower, decorativ...","[cardinal larkspur, tall bellflower, decorativ..."


In [35]:
results_two['predicted_top_10'][4]

['tree heath',
 'grand fir',
 'chehalis apple',
 'hornbeam maple',
 'silver birch',
 'mountain bush honeysuckle',
 'sourwood',
 'small-leaved cotoneaster',
 'rain tree',
 'hazel alder']